## [Optimization](https://en.wikipedia.org/wiki/Quantum_optimization_algorithms) & [Variational](https://arxiv.org/pdf/2012.09265) (Hybrid) Algorithms

The [Noisy Intermediate-Scale Quantum (NISQ)](https://en.wikipedia.org/wiki/Noisy_intermediate-scale_quantum_computing) era is defined by quantum processors with limited qubit counts, lack of full quantum error correction, and short coherence times. [Hybrid quantum-classical algorithms](https://arxiv.org/pdf/2207.06850) address these hardware limitations by splitting the workload: a shallow parameterized quantum circuit executes state preparation and measurements, while an HPC or classical processor handles parameter optimization.

                 +-----------------------------------+              
                 |        Classical Optimizer        |              
                 |   (e.g., COBYLA, SPSA, Adam)      |              
                 +-----------------+-----------------+              
                                   |                                
                       Updates Parameters (θ_i)                     
                                   |                                
                                   v                                
                 +-----------------------------------+              
                 |   Parameterized Quantum Circuit   |              
                 |      (Ansatz State Execution)     |              
                 +-----------------------------------+              

### 1. [Variational Quantum Eigensolver (VQE)](https://en.wikipedia.org/wiki/Variational_quantum_eigensolver)

* **Problem Domain:** Quantum Chemistry, Condensed Matter Physics, Material Science.
* **Core Function:** Estimates the ground-state energy (lowest eigenvalue) of a system's molecular or physical Hamiltonian ($H$).  
* **Algorithmic Mechanism:**
    1. The target Hamiltonian $H$ is mapped into a linear combination of Pauli operators ($H = \sum_i c_i P_i$).
    2. A parameterized quantum circuit (ansatz) prepares a quantum state $\vert{}\psi(\theta)\rangle$.
    3. Quantum hardware measures expectation values $E(\theta) = \langle\psi(\theta)\vert{} H \vert{}\psi(\theta)\rangle$.
    4. A classical optimizer uses the Rayleigh-Ritz Variational Principle ($E(\theta) \ge E_0$) to update parameters $\theta$ until $E(\theta)$ converges to the ground state.
    
* **Complexity & Scale:** 
    * **Quantum:** Circuit depth scales as $\mathcal{O}(\text{poly}(\log N))$.
    * **Classical/HPC:** Runtime is bound by sampling noise $\mathcal{O}(1/\varepsilon^2)$ and classical optimization iterations.

**Example:** 

It finds the ground-state energy of a simple 2-qubit transverse-field Ising Hamiltonian ($H = Z_0 Z_1 + X_0$) using a parameterized ansatz circuit.

In [4]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector

# 1. Define target Hamiltonian: H = Z_0 Z_1 + X_0
hamiltonian = SparsePauliOp(["ZZ", "XI"], coeffs=[1.0, 1.0])

# 2. Construct a Parameterized Quantum Circuit (Ansatz)
theta = ParameterVector('θ', 3)
ansatz = QuantumCircuit(2)
ansatz.ry(theta[0], 0)
ansatz.ry(theta[1], 1)
ansatz.cx(0, 1)
ansatz.ry(theta[2], 0)

# 3. Objective function evaluated classically
def vqe_cost_function(params):
    # Bind parameters to circuit
    bound_circuit = ansatz.assign_parameters(params)
    
    # Compute statevector and calculate expectation value <ψ(θ)| H |ψ(θ)>
    state = Statevector(bound_circuit)
    energy = state.expectation_value(hamiltonian)
    
    return np.real(energy)

# Example evaluation at initial trial parameters
initial_params = [0.1, 0.2, 0.3]
print("Initial Energy Evaluation:", vqe_cost_function(initial_params))
print("\nVQE Ansatz Circuit:")
print(ansatz.draw('text'))

Initial Energy Evaluation: 1.1349626943792603

VQE Ansatz Circuit:
     ┌──────────┐     ┌──────────┐
q_0: ┤ Ry(θ[0]) ├──■──┤ Ry(θ[2]) ├
     ├──────────┤┌─┴─┐└──────────┘
q_1: ┤ Ry(θ[1]) ├┤ X ├────────────
     └──────────┘└───┘            


### 2. [Quantum Approximate Optimization Algorithm (QAOA)](https://arxiv.org/pdf/2511.18377)

* **Problem Domain:** [Combinatorial Optimization](https://en.wikipedia.org/wiki/Combinatorial_optimization), [Graph Theory](https://en.wikipedia.org/wiki/Graph_theory) (e.g., [Max-Cut](https://en.wikipedia.org/wiki/Maximum_cut), [TSP](https://en.wikipedia.org/wiki/Travelling_salesman_problem), [Portfolio Optimization](https://www.nature.com/articles/s41598-023-45392-w)).  
* **Core Function:** Finds approximate solutions to NP-hard and NP-complete combinatorial optimization problems.
* **Algorithmic Mechanism:**
    1. Translates a classical cost function into a target Problem Hamiltonian $H_C$.
    2. Alternates between applying $H_C$ and a Mixer Hamiltonian $H_M$ across $p$ layers via parameterized angles $(\gamma_1 \dots \gamma_p, \beta_1 \dots \beta_p)$:
    $$
        \vert{}\psi(\gamma, \beta)\rangle = \prod_{k=1}^p e^{-i \beta_k H_M} e^{-i \gamma_k H_C} \vert{}+\rangle^{\otimes n}
    $$
    3. The quantum device measures computational basis states to evaluate cost expectations.
    4. The classical loop optimizes the angle parameters to maximize expected solution quality.
    
* **Complexity & Scale:**
    * **Quantum:** Circuit depth scales linearly with layer depth $\mathcal{O}(p)$.
    * **Classical/HPC:** Scalability depends on the classical optimizer navigating non-convex parameter landscapes (avoiding barren plateaus).

**Example:**

Constructing a 1-layer ($p=1$) QAOA circuit for a 2-node Max-Cut graph problem with cost Hamiltonian $H_C = Z_0 Z_1$ and standard mixer $H_M = X_0 + X_1$.

In [2]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# QAOA parameters: gamma (cost) and beta (mixer)
gamma = Parameter('γ')
beta = Parameter('β')

qc = QuantumCircuit(2)

# Step 1: Initial state preparation (Equal superposition |+>)
qc.h([0, 1])

# Step 2: Cost Unitary exp(-i * gamma * Z_0 Z_1)
qc.rzz(2 * gamma, 0, 1)

# Step 3: Mixer Unitary exp(-i * beta * (X_0 + X_1))
qc.rx(2 * beta, 0)
qc.rx(2 * beta, 1)

# Measure in computational basis
qc.measure_all()

# Bind parameters to specific values for execution
bound_qaoa = qc.assign_parameters({gamma: 0.3927, beta: 0.7854})

print("1-Layer QAOA Circuit (Max-Cut):")
print(bound_qaoa.draw('text'))

1-Layer QAOA Circuit (Max-Cut):
        ┌───┐             ┌────────────┐ ░ ┌─┐   
   q_0: ┤ H ├─■───────────┤ Rx(1.5708) ├─░─┤M├───
        ├───┤ │ZZ(0.7854) ├────────────┤ ░ └╥┘┌─┐
   q_1: ┤ H ├─■───────────┤ Rx(1.5708) ├─░──╫─┤M├
        └───┘             └────────────┘ ░  ║ └╥┘
meas: 2/════════════════════════════════════╩══╩═
                                            0  1 


### 3. [Quantum Annealing](https://en.wikipedia.org/wiki/Quantum_annealing) & [Adiabatic Quantum Computation (AQC)](https://en.wikipedia.org/wiki/Adiabatic_quantum_computation)
* **Problem Domain:** [Discrete Binary Optimization](https://mdobook.github.io/html/discrete/), [Ising Spin Glasses](https://en.wikipedia.org/wiki/Spin_glass), [Quadratic Unconstrained Binary Optimization (QUBO)](https://en.wikipedia.org/wiki/Quadratic_unconstrained_binary_optimization).  
* **Core Function:** Finds the global minimum energy state of complex combinatorial loss functions using continuous physical dynamics rather than gate-based circuits. 
* **Algorithmic Mechanism:**
    1. Prepares the system in the simple ground state of an initial Hamiltonian $H_0$.  
    2. Slowly evolves the system over time $t \in [0, T]$ according to $H(t) = (1 - s(t))H_0 + s(t)H_{\text{problem}}$.  
    3. According to the [Adiabatic Theorem](https://en.wikipedia.org/wiki/Adiabatic_theorem), if evolution is sufficiently slow, the system remains in the ground state, yielding the optimal solution to $H_{\text{problem}}$ at $t = T$.  
    
* **Complexity & Scale:**
    * System runtime is governed by the minimum spectral gap $\Delta_{min}$ between the ground state and first excited state: $\mathcal{O}(1/\Delta_{min}^2)$.

**Example:**

While Quantum Annealers (such as D-Wave) execute continuous physical state evolution rather than gate-based operations, we can emulate an adiabatic annealing path on a gate-based quantum system by discretization using a time-dependent schedule $s(t) \in [0, 1]$.

In [3]:
import numpy as np
from qiskit import QuantumCircuit

def build_annealing_step(s, dt):
    """
    Simulates a single step s in adiabatic evolution:
    H(s) = (1 - s) * H_mixer + s * H_problem
    where H_mixer = X_0 + X_1 and H_problem = Z_0 Z_1
    """
    qc = QuantumCircuit(2)
    
    # Apply driver/mixer field weighted by (1 - s)
    qc.rx(2 * (1 - s) * dt, 0)
    qc.rx(2 * (1 - s) * dt, 1)
    
    # Apply problem coupling weighted by s
    qc.rzz(2 * s * dt, 0, 1)
    
    return qc

# Simulate adiabatic schedule s: 0.0 -> 1.0 across 4 discrete annealing steps
steps = 4
dt = 0.5
annealing_circuit = QuantumCircuit(2)

# Prepare initial ground state of Mixer (|++>)
annealing_circuit.h([0, 1])

# Sweep schedule parameter s from 0 to 1
for step in range(steps):
    s = (step + 1) / steps
    step_circuit = build_annealing_step(s, dt)
    annealing_circuit.compose(step_circuit, inplace=True)

print("Discretized Annealing Emulation Circuit:")
print(annealing_circuit.draw('text'))

Discretized Annealing Emulation Circuit:
     ┌───┐┌──────────┐           ┌─────────┐          ┌──────────┐           »
q_0: ┤ H ├┤ Rx(0.75) ├─■─────────┤ Rx(0.5) ├─■────────┤ Rx(0.25) ├─■─────────»
     ├───┤├──────────┤ │ZZ(0.25) ├─────────┤ │ZZ(0.5) ├──────────┤ │ZZ(0.75) »
q_1: ┤ H ├┤ Rx(0.75) ├─■─────────┤ Rx(0.5) ├─■────────┤ Rx(0.25) ├─■─────────»
     └───┘└──────────┘           └─────────┘          └──────────┘           »
«     ┌───────┐        
«q_0: ┤ Rx(0) ├─■──────
«     ├───────┤ │ZZ(1) 
«q_1: ┤ Rx(0) ├─■──────
«     └───────┘        


**Key Algorithmic Trade-offs**

| Feature | Gate-Based Hybrid (VQE / QAOA) | Quantum Annealing |
| :--- | :--- | :--- |
| **Hardware Architecture** | Universal Gate-Based NISQ Processors | Special-Purpose Annealing Systems (e.g., D-Wave) |
| **Control Mechanism** | Discrete Parameterized Gates ($U(\theta)$) | Continuous Global Magnetic/Coupling Shifts |
| **Optimization Boundary** | Hybrid Quantum-Classical Feedback Loop | Physical Quantum Tunneling & Adiabatic Decay |